# Heart Disease Prediction — Ensemble Notebook (v2: Wall-Breaker)

**Competition:** Playground Series S6E2  
**Approach:** CatBoost × RealMLP × XGBoost → Weighted Rank Blend + Local Correction + Soft Noise Weighting

| | Score |
|---|---|
| **CV (OOF AUC) v1** | 0.955720 |
| **Public LB v1** | 0.95396 |

### What's new in v2
- **Phase 1A**: OOF correlation & hard-example AUC analysis
- **Phase 1B**: Weighted rank blend grid search (OOF-tuned weights)
- **Phase 1C**: Local correction for `cp=ASY` subgroup
- **Phase 2A**: Soft noise weighting (sample weight downweighting of noisy examples)
- **Phase 2B**: XGBoost model added for diversity
- **Phase 2C**: 3-model weighted blend with OOF-tuned weights
- **Phase 3**: Regime-aware evaluation (ASY / silent high-risk / structural groups)


## 1. Environment & Installs

In [ ]:
!pip install pytabkit catboost xgboost -q

## 2. Imports & Global Config

In [ ]:
# Standard library
import json
import subprocess
import warnings
import zipfile
from pathlib import Path

# Numerical / data
import numpy as np
import pandas as pd

# Machine learning
import torch
from catboost import CatBoostClassifier, Pool
from pytabkit import RealMLP_TD_Classifier
from scipy.stats import rankdata, spearmanr
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

warnings.filterwarnings('ignore')

# Global config
CFG = dict(
    comp_slug       = "playground-series-s6e2",
    target_col      = "Heart Disease",
    id_col          = "id",
    n_folds         = 5,
    random_state    = 42,
    # Paths
    kaggle_comp_dir = Path("/kaggle/input/competitions/playground-series-s6e2"),
    kaggle_ext_path = Path("/kaggle/input/datasets/neurocipher/heartdisease/Heart_Disease_Prediction.csv"),
    local_data_dir  = Path("data/raw"),
    need_files      = ["train.csv", "test.csv", "sample_submission.csv"],
    # Soft noise weighting
    noise_alpha     = 0.25,   # 0.15 ~ 0.35: downweight scale for noisy samples
    noise_min_w     = 0.50,   # floor for sample weight
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(CFG['random_state'])
np.random.seed(CFG['random_state'])
if DEVICE == 'cuda':
    torch.cuda.manual_seed_all(CFG['random_state'])
    torch.set_float32_matmul_precision('high')

CFG['local_data_dir'].mkdir(parents=True, exist_ok=True)
print(f"Device: {DEVICE}")


## 3. Utilities

In [ ]:
# ── Data access helpers ─────────────────────────────────────────────────────

def _run(cmd):
    subprocess.run(cmd, capture_output=True, text=True, check=True)

def _ensure_kaggle_json_colab(dst=Path("/content/kaggle.json")):
    if dst.exists():
        print("Found:", dst); return dst
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f"{dst} not found.")
    print("Upload your kaggle.json")
    uploaded = files.upload()
    cand = next((n for n in uploaded if n.endswith(".json")), None)
    if cand is None:
        raise FileNotFoundError("Upload failed.")
    Path(cand).rename(dst)
    return dst

def _install_kaggle_json(src):
    dst_dir = Path.home() / ".kaggle"
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / "kaggle.json"
    dst.write_bytes(src.read_bytes())
    try: dst.chmod(0o600)
    except Exception: pass

def _local_data_ready(data_dir):
    return all((data_dir / f).exists() for f in CFG['need_files'])

def _download_competition(data_dir):
    _run(["kaggle", "config", "view"])
    _run(["kaggle", "competitions", "download", "-c", CFG['comp_slug'], "-p", str(data_dir), "--force"])
    zips = list(data_dir.glob("*.zip"))
    for zp in zips:
        with zipfile.ZipFile(zp, "r") as zf:
            zf.extractall(data_dir)
        print("Unzipped:", zp.name)

# ── Preprocessing helpers ───────────────────────────────────────────────────

def encode_target_strict(y):
    mapping_candidates = [
        {"No": 0, "Yes": 1}, {"N": 0, "Y": 1},
        {"Negative": 0, "Positive": 1}, {"Absent": 0, "Present": 1},
        {"Absence": 0, "Presence": 1}, {0: 0, 1: 1}, {"0": 0, "1": 1},
    ]
    uniq = set(pd.Series(y).dropna().unique().tolist())
    for mp in mapping_candidates:
        if uniq.issubset(set(mp.keys())):
            return pd.Series(y).map(mp).astype("int8")
    raise ValueError(f"Unknown target labels: {sorted(list(uniq))}")

def check_data_quality(df, name="Dataset"):
    cols = [c for c in df.columns if c != 'id']
    dupes = df.duplicated(subset=cols).sum()
    nans  = df.isnull().sum().sum()
    print(f"[{name}] rows={len(df)}, duplicates={dupes}, total_NaN={nans}")

def add_external_target_stats(df, original_df, base_features, target_col):
    if original_df is None:
        return df.copy()
    out = df.copy()
    n = len(out)
    global_mean   = float(original_df[target_col].mean())
    global_median = float(original_df[target_col].median())
    for col in base_features:
        if col not in original_df.columns:
            continue
        stats = (
            original_df.groupby(col)[target_col]
            .agg(["mean", "median", "std", "skew", "count"])
            .reset_index()
        )
        stats.columns = [col] + [f"orig_{col}_{s}" for s in ["mean", "median", "std", "skew", "count"]]
        out = out.merge(stats, on=col, how="left")
        assert len(out) == n
        out = out.fillna({
            f"orig_{col}_mean":   global_mean,
            f"orig_{col}_median": global_median,
            f"orig_{col}_std":    0.0,
            f"orig_{col}_skew":   0.0,
            f"orig_{col}_count":  0.0,
        })
    return out

# ── Ensemble helpers ────────────────────────────────────────────────────────

def rank01(x):
    """Map array to [0,1] via average-rank normalisation."""
    x = np.asarray(x, dtype=np.float64)
    r = rankdata(x, method="average")
    return (r - 0.5) / len(r)

def weighted_rank_blend(oofs: dict, weights: dict) -> np.ndarray:
    """Weighted blend of rank-normalised OOF predictions."""
    total = sum(weights[k] for k in oofs)
    out = sum(weights[k] * rank01(oofs[k]) for k in oofs)
    return out / total

def auc_subgroup(y, pred, mask):
    """AUC on a boolean mask. Returns None if mask has only 1 class."""
    if mask.sum() < 5:
        return None
    try:
        return roc_auc_score(y[mask], pred[mask])
    except Exception:
        return None

def make_soft_weights(oof_blend, y, alpha=0.25, min_w=0.5):
    """
    Compute per-sample soft weights based on |oof_blend - y|.
    Noisier samples get lower weight (down to min_w).
    """
    noise = np.abs(oof_blend - y.astype(float))
    noise_norm = (noise - noise.min()) / (noise.max() - noise.min() + 1e-9)
    w = 1.0 - alpha * noise_norm
    return np.clip(w, min_w, 1.0).astype(np.float32)

def print_auc_table(y, oof_dict, subgroup_masks=None):
    """Print AUC table for multiple OOF preds, with optional subgroup columns."""
    headers = ["Model", "All-AUC"]
    if subgroup_masks:
        headers += list(subgroup_masks.keys())
    rows = []
    for name, oof in oof_dict.items():
        row = [name, f"{roc_auc_score(y, oof):.5f}"]
        if subgroup_masks:
            for sg_name, mask in subgroup_masks.items():
                v = auc_subgroup(y, oof, mask)
                row.append(f"{v:.5f}" if v else "N/A")
        rows.append(row)
    col_w = [max(len(h), max(len(r[i]) for r in rows)) for i, h in enumerate(headers)]
    sep = " | ".join("-" * w for w in col_w)
    hdr = " | ".join(h.ljust(col_w[i]) for i, h in enumerate(headers))
    print(hdr); print(sep)
    for row in rows:
        print(" | ".join(row[i].ljust(col_w[i]) for i in range(len(headers))))


## 4. Data Access

In [ ]:
kaggle_dir = CFG['kaggle_comp_dir']
local_dir  = CFG['local_data_dir']

if kaggle_dir.exists():
    data_dir = kaggle_dir
    print("[Kaggle] Using mounted competition data:", data_dir)
else:
    data_dir = local_dir
    if _local_data_ready(data_dir):
        print("[Local] Data already present:", data_dir)
    else:
        print("[Download] Fetching via Kaggle API...")
        kaggle_json = _ensure_kaggle_json_colab(Path("/content/kaggle.json"))
        _install_kaggle_json(kaggle_json)
        _download_competition(data_dir)
        print("Download complete.")


## 5. Data Loading

In [ ]:
for fname in CFG['need_files']:
    if not (data_dir / fname).exists():
        raise FileNotFoundError(f"Required file missing: {data_dir / fname}")

train = pd.read_csv(data_dir / "train.csv")
test  = pd.read_csv(data_dir / "test.csv")
sub   = pd.read_csv(data_dir / "sample_submission.csv")

ext_path = CFG['kaggle_ext_path']
original = pd.read_csv(ext_path) if ext_path.exists() else None

print(f"train: {train.shape}, test: {test.shape}")
print(f"external: {'loaded' if original is not None else 'not available'}")
check_data_quality(train, "Train")
check_data_quality(test,  "Test")


## 6. Preprocessing & Feature Engineering

In [ ]:
TARGET_COL = CFG['target_col']
ID_COL     = CFG['id_col']
META_COLS  = [TARGET_COL, ID_COL, "is_external"]

# 1) Encode target
train_comp = train.copy()
if not pd.api.types.is_numeric_dtype(train_comp[TARGET_COL]):
    le = LabelEncoder()
    train_comp[TARGET_COL] = le.fit_transform(train_comp[TARGET_COL])
    if original is not None and TARGET_COL in original.columns:
        original[TARGET_COL] = le.transform(original[TARGET_COL])

# 2) Align and append external data
if original is not None:
    if not pd.api.types.is_numeric_dtype(original[TARGET_COL]):
        original[TARGET_COL] = encode_target_strict(original[TARGET_COL])
    orig_aligned = original.copy()
    for col in train_comp.columns:
        if col not in orig_aligned.columns:
            orig_aligned[col] = np.nan
    if ID_COL not in orig_aligned.columns:
        orig_aligned[ID_COL] = -(np.arange(len(orig_aligned)) + 1)
    orig_aligned = orig_aligned[train_comp.columns]
    train_full = pd.concat([train_comp, orig_aligned], ignore_index=True)
    train_full["is_external"] = [0] * len(train_comp) + [1] * len(orig_aligned)
else:
    train_full = train_comp.copy()
    train_full["is_external"] = 0

train = train_full
print(f"Combined train: {train.shape}")

BASE_FEATURES = [c for c in train.columns if c not in META_COLS]

# 3) External target stats
train_fe = add_external_target_stats(train, original, BASE_FEATURES, TARGET_COL)
test_fe  = add_external_target_stats(test,  original, BASE_FEATURES, TARGET_COL)

# 4) Build X / y / X_test
X = train_fe.drop(columns=[c for c in META_COLS if c in train_fe.columns])
y = train_fe[TARGET_COL].astype("int8")
X_test = test_fe.drop(columns=[c for c in [ID_COL] if c in test_fe.columns])

# Cast all to string category
for col in X.columns:
    X[col]      = X[col].astype(str).astype("category")
    X_test[col] = X_test[col].astype(str).astype("category")

cat_cols = list(X.columns)
assert len(X) == len(y)
assert len(X_test) == len(test)
print(f"X: {X.shape}  |  X_test: {X_test.shape}")
print(f"Columns: {list(X.columns)}")


## 7. Validation Setup

In [ ]:
N_FOLDS      = CFG['n_folds']
RANDOM_STATE = CFG['random_state']

comp_mask = (train["is_external"] == 0).values

X_comp    = X.loc[comp_mask].reset_index(drop=True)
y_comp    = y.loc[comp_mask].reset_index(drop=True)
comp_ids  = train.loc[comp_mask, ID_COL].reset_index(drop=True)

X_test_comp = X_test.copy()
test_ids    = test[ID_COL].reset_index(drop=True)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

print(f"X_comp: {X_comp.shape}  |  y_comp distribution: {dict(y_comp.value_counts().sort_index())}")
print(f"X_test_comp: {X_test_comp.shape}")
print(f"CV: {N_FOLDS}-fold stratified, seed={RANDOM_STATE}")


## 8. Subgroup Masks

Define subgroups for regime-aware evaluation.  
Column names are normalised — adjust `_CP_COL`, `_OP_COL`, etc. if the dataset uses different names.


In [ ]:
# ── Detect key column names (case-insensitive) ──────────────────────────────
def _find_col(candidates, columns):
    """Return first match from candidates (case-insensitive), else None."""
    cols_lower = {c.lower(): c for c in columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    return None

_CP_COL  = _find_col(["ChestPainType", "cp", "chest_pain"], X_comp.columns)
_OP_COL  = _find_col(["Oldpeak", "oldpeak", "ST_Slope"], X_comp.columns)
_CA_COL  = _find_col(["CA", "ca", "NumMajorVessels"], X_comp.columns)
_TH_COL  = _find_col(["Thal", "thal", "Thallium"], X_comp.columns)
_EX_COL  = _find_col(["ExerciseAngina", "exang", "exercise_angina"], X_comp.columns)
_HR_COL  = _find_col(["MaxHR", "maxhr", "max_heart_rate", "MaxHeartRate"], X_comp.columns)

print("Detected columns:")
for name, col in [("ChestPainType", _CP_COL), ("Oldpeak", _OP_COL),
                  ("CA", _CA_COL), ("Thal", _TH_COL),
                  ("ExerciseAngina", _EX_COL), ("MaxHR", _HR_COL)]:
    print(f"  {name:<20} → {col}")

# ── Build boolean masks ──────────────────────────────────────────────────────
def _str_mask(col_name, values):
    """Mask where column (string category) is in values."""
    if col_name is None:
        return np.zeros(len(X_comp), dtype=bool)
    col = X_comp[col_name].astype(str).str.strip()
    str_vals = [str(v) for v in values]
    return col.isin(str_vals).values

# Regime A: Asymptomatic chest pain (cp=ASY / 4)
mask_asy   = _str_mask(_CP_COL, ["ASY", "4", "asymptomatic"])

# Regime B: Oldpeak = 0 (silent stress response)
mask_op0   = _str_mask(_OP_COL, ["0", "0.0"])

# Regime C: High vessel count (ca > 0)
mask_ca_pos = ~_str_mask(_CA_COL, ["0", "0.0"]) if _CA_COL else np.zeros(len(X_comp), dtype=bool)

# Silent high-risk: cp=ASY AND (ca>0 OR thal abnormal)
_thal_abn  = _str_mask(_TH_COL, ["2", "3", "reversible defect", "fixed defect", "RD", "FD"])
mask_silent_hr = mask_asy & (mask_ca_pos | _thal_abn)

# Hard examples: |blend - y| top 8%  — will be filled after initial blend
# (placeholder — updated in Phase 1A)
mask_hard = np.zeros(len(X_comp), dtype=bool)

subgroup_masks = {
    "cp=ASY":     mask_asy,
    "op=0":       mask_op0,
    "ca>0":       mask_ca_pos,
    "silent_hr":  mask_silent_hr,
}

for sg, m in subgroup_masks.items():
    pos = y_comp[m].sum() if m.any() else 0
    print(f"  {sg:<15}: n={m.sum():4d}  pos={pos}  ({m.mean()*100:.1f}%)")


## 9. CatBoost Training — Round 1 (Standard)

In [ ]:
cat_features_idx = [X_comp.columns.get_loc(c) for c in cat_cols if c in X_comp.columns]

oof_cat  = np.zeros(len(X_comp), dtype="float32")
test_cat = np.zeros(len(X_test_comp), dtype="float32")

_CAT_PARAMS = dict(
    loss_function     = "Logloss",
    eval_metric       = "AUC",
    learning_rate     = 0.03,
    depth             = 8,
    l2_leaf_reg       = 10,
    random_strength   = 1.0,
    bagging_temperature = 0.5,
    border_count      = 128,
    iterations        = 4000,
    od_type           = "Iter",
    od_wait           = 200,
    verbose           = 200,
    task_type         = "GPU" if DEVICE == "cuda" else "CPU",
)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_comp, y_comp), start=1):
    print(f"[CatBoost R1] Fold {fold}/{N_FOLDS}")
    X_tr, X_val = X_comp.iloc[tr_idx], X_comp.iloc[val_idx]
    y_tr, y_val = y_comp.iloc[tr_idx], y_comp.iloc[val_idx]

    train_pool = Pool(X_tr, label=y_tr, cat_features=cat_features_idx)
    valid_pool = Pool(X_val, label=y_val, cat_features=cat_features_idx)
    test_pool  = Pool(X_test_comp, cat_features=cat_features_idx)

    model = CatBoostClassifier(random_seed=RANDOM_STATE + fold, **_CAT_PARAMS)
    model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

    oof_cat[val_idx]  = model.predict_proba(valid_pool)[:, 1].astype("float32")
    test_cat         += (model.predict_proba(test_pool)[:, 1] / N_FOLDS).astype("float32")

cat_oof_auc = roc_auc_score(y_comp, oof_cat)
print(f"\n[CatBoost R1] OOF AUC = {cat_oof_auc:.6f}")


## 10. RealMLP Training

In [ ]:
REALMLP_PARAMS = {
    'random_state': RANDOM_STATE,
    'verbosity': 2,
    'n_epochs': 100,
    'batch_size': 2**12,
    'n_ens': 8,
    'use_early_stopping': True,
    'early_stopping_additive_patience': 20,
    'early_stopping_multiplicative_patience': 1,
    'act': "mish",
    'embedding_size': 8,
    'first_layer_lr_factor': 0.5962121993798933,
    'val_metric_name': '1-auc_ovr',
    'hidden_sizes': "rectangular",
    'hidden_width': 384,
    'lr': 0.04,
    'ls_eps': 0.011498317194338772,
    'ls_eps_sched': "coslog4",
    'max_one_hot_cat_size': 18,
    'n_hidden_layers': 4,
    'p_drop': 0.07301419697186451,
    'p_drop_sched': "flat_cos",
    'plr_hidden_1': 16,
    'plr_hidden_2': 8,
    'plr_lr_factor': 0.1151437622270563,
    'plr_sigma': 2.3316811282666916,
    'scale_lr_factor': 2.244801835541429,
    'sq_mom': 1.0 - 0.011834054955582318,
    'wd': 0.02369230879235962,
    'device': DEVICE,
}

def make_realmlp(params, cat_col_names):
    try:
        return RealMLP_TD_Classifier(**params, cat_col_names=cat_col_names)
    except TypeError:
        return RealMLP_TD_Classifier(**params)

oof_realmlp  = np.zeros(len(X_comp), dtype="float32")
test_realmlp = np.zeros(len(X_test_comp), dtype="float32")

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_comp, y_comp), start=1):
    print(f"\n--- RealMLP Fold {fold}/{N_FOLDS} ---")
    X_tr, X_val = X_comp.iloc[tr_idx], X_comp.iloc[val_idx]
    y_tr, y_val = y_comp.iloc[tr_idx], y_comp.iloc[val_idx]

    model = make_realmlp(REALMLP_PARAMS, cat_cols)
    model.fit(X_tr, y_tr.values, X_val, y_val.values)

    oof_realmlp[val_idx]  = model.predict_proba(X_val)[:, 1].astype("float32")
    test_realmlp         += (model.predict_proba(X_test_comp)[:, 1] / N_FOLDS).astype("float32")

    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

print(f"\n[RealMLP] OOF AUC = {roc_auc_score(y_comp, oof_realmlp):.6f}")


## 11. XGBoost Training

In [ ]:
# XGBoost needs numeric input — label-encode all categorical columns
from sklearn.preprocessing import OrdinalEncoder

_oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_comp_num    = pd.DataFrame(_oe.fit_transform(X_comp.astype(str)),
                              columns=X_comp.columns, dtype=np.float32)
X_test_num    = pd.DataFrame(_oe.transform(X_test_comp.astype(str)),
                              columns=X_test_comp.columns, dtype=np.float32)

_XGB_PARAMS = dict(
    n_estimators      = 4000,
    learning_rate     = 0.03,
    max_depth         = 6,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    eval_metric       = "auc",
    tree_method       = "hist",
    device            = "cuda" if DEVICE == "cuda" else "cpu",
    random_state      = RANDOM_STATE,
    early_stopping_rounds = 200,
    verbosity         = 0,
)

oof_xgb  = np.zeros(len(X_comp), dtype="float32")
test_xgb = np.zeros(len(X_test_num), dtype="float32")

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_comp_num, y_comp), start=1):
    print(f"[XGBoost] Fold {fold}/{N_FOLDS}")
    X_tr, X_val = X_comp_num.iloc[tr_idx], X_comp_num.iloc[val_idx]
    y_tr, y_val = y_comp.iloc[tr_idx], y_comp.iloc[val_idx]

    model = xgb.XGBClassifier(**_XGB_PARAMS)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    oof_xgb[val_idx]  = model.predict_proba(X_val)[:, 1].astype("float32")
    test_xgb         += (model.predict_proba(X_test_num)[:, 1] / N_FOLDS).astype("float32")

print(f"\n[XGBoost] OOF AUC = {roc_auc_score(y_comp, oof_xgb):.6f}")


## 12. Phase 1A — OOF Correlation & Hard-Example AUC Analysis

> **Purpose**: Check model diversity (Spearman r) and confirm whether models differ on hard examples.  
> r > 0.99 → models are nearly identical → prioritise diversity over blending.


In [ ]:
print("="*60)
print("Phase 1A: OOF Correlation & Hard-Example Analysis")
print("="*60)

# ── Spearman correlations ────────────────────────────────────────────────────
pairs = [("Cat", oof_cat), ("RealMLP", oof_realmlp), ("XGB", oof_xgb)]
print("\nSpearman r (OOF predictions):")
for i, (na, a) in enumerate(pairs):
    for j, (nb, b) in enumerate(pairs):
        if j > i:
            r, p = spearmanr(a, b)
            print(f"  {na:<10} vs {nb:<10}: r={r:.4f}  (p={p:.2e})")

# ── Initial equal-weight blend for hard-example detection ──────────────────
oof_blend_init = np.mean([rank01(oof_cat), rank01(oof_realmlp), rank01(oof_xgb)], axis=0)

# ── Hard example mask (top 8% residual) ──────────────────────────────────────
residual    = np.abs(oof_blend_init - y_comp.values.astype(float))
hard_thresh = np.percentile(residual, 92)
mask_hard   = residual >= hard_thresh  # update global mask

subgroup_masks["hard_8pct"] = mask_hard
print(f"\nHard examples (top 8% residual): n={mask_hard.sum()}  ({mask_hard.mean()*100:.1f}%)")

# ── AUC table across all subgroups ───────────────────────────────────────────
print()
oof_dict = {
    "CatBoost": oof_cat,
    "RealMLP":  oof_realmlp,
    "XGBoost":  oof_xgb,
    "Blend(eq)": oof_blend_init,
}
print_auc_table(y_comp, oof_dict, subgroup_masks)


## 13. Phase 1B — Weighted Rank Blend Grid Search

Grid search over 3-model blend weights on OOF to find the combination that maximises overall AUC.


In [ ]:
print("="*60)
print("Phase 1B: Weighted Rank Blend Grid Search")
print("="*60)

r_cat    = rank01(oof_cat)
r_mlp    = rank01(oof_realmlp)
r_xgb    = rank01(oof_xgb)

best_auc  = 0
best_ws   = (1/3, 1/3, 1/3)
results   = []

step = 0.05
ws = np.arange(0.0, 1.01, step)

for wc in ws:
    for wm in ws:
        wx = round(1.0 - wc - wm, 8)
        if wx < -1e-6 or wx > 1.0 + 1e-6:
            continue
        wx = max(0.0, min(1.0, wx))
        blend = wc * r_cat + wm * r_mlp + wx * r_xgb
        auc = roc_auc_score(y_comp, blend)
        results.append((auc, wc, wm, wx))
        if auc > best_auc:
            best_auc = auc
            best_ws = (wc, wm, wx)

# Sort top 10
results.sort(reverse=True)
print("\nTop 10 blends (Cat | MLP | XGB):")
for auc, wc, wm, wx in results[:10]:
    print(f"  AUC={auc:.6f}  Cat={wc:.2f}  MLP={wm:.2f}  XGB={wx:.2f}")

BEST_WC, BEST_WM, BEST_WX = best_ws
print(f"\n★ Best weights:  Cat={BEST_WC:.2f}  MLP={BEST_WM:.2f}  XGB={BEST_WX:.2f}  →  OOF AUC={best_auc:.6f}")
print(f"  Baseline (equal): {roc_auc_score(y_comp, oof_blend_init):.6f}")

# Build best OOF and test blends
oof_blend_opt  = BEST_WC * r_cat + BEST_WM * r_mlp + BEST_WX * r_xgb
r_cat_test     = rank01(test_cat)
r_mlp_test     = rank01(test_realmlp)
r_xgb_test     = rank01(test_xgb)
test_blend_opt = BEST_WC * r_cat_test + BEST_WM * r_mlp_test + BEST_WX * r_xgb_test


## 14. Phase 1C — Local Correction (cp=ASY Subgroup)

Asymptomatic chest pain (`cp=ASY`) is the hardest subgroup.  
We search for the best per-model weight within this group independently.


In [ ]:
print("="*60)
print("Phase 1C: Local Correction — cp=ASY subgroup")
print("="*60)

oof_blend_lc  = oof_blend_opt.copy()
test_blend_lc = test_blend_opt.copy()

_do_local = mask_asy.any() and y_comp[mask_asy].nunique() > 1

if not _do_local:
    print("cp=ASY mask is empty or single-class — skipping local correction.")
else:
    # Search best (Cat, MLP, XGB) weights for ASY subgroup
    best_lc_auc = roc_auc_score(y_comp, oof_blend_opt)
    best_lc_ws  = (BEST_WC, BEST_WM, BEST_WX)

    for wc in ws:
        for wm in ws:
            wx = round(1.0 - wc - wm, 8)
            if wx < -1e-6 or wx > 1.0 + 1e-6:
                continue
            wx = max(0.0, min(1.0, wx))

            # Candidate: replace ASY predictions with new blend
            oof_tmp = oof_blend_opt.copy()
            oof_tmp[mask_asy] = (wc * r_cat[mask_asy]
                                 + wm * r_mlp[mask_asy]
                                 + wx * r_xgb[mask_asy])
            auc = roc_auc_score(y_comp, oof_tmp)
            if auc > best_lc_auc:
                best_lc_auc = auc
                best_lc_ws = (wc, wm, wx)

    LC_WC, LC_WM, LC_WX = best_lc_ws
    print(f"Local correction weights (cp=ASY):  Cat={LC_WC:.2f}  MLP={LC_WM:.2f}  XGB={LC_WX:.2f}")
    print(f"OOF AUC after local correction: {best_lc_auc:.6f}  (before: {roc_auc_score(y_comp, oof_blend_opt):.6f})")

    # Apply to OOF
    oof_blend_lc[mask_asy] = (LC_WC * r_cat[mask_asy]
                               + LC_WM * r_mlp[mask_asy]
                               + LC_WX * r_xgb[mask_asy])

    # Apply to test
    mask_asy_test = _str_mask(_CP_COL, ["ASY", "4", "asymptomatic"]) if _CP_COL else np.zeros(len(X_test_comp), dtype=bool)
    # Recompute _str_mask on test
    def _str_mask_test(col_name, values):
        if col_name is None:
            return np.zeros(len(X_test_comp), dtype=bool)
        col = X_test_comp[col_name].astype(str).str.strip()
        return col.isin([str(v) for v in values]).values
    mask_asy_test = _str_mask_test(_CP_COL, ["ASY", "4", "asymptomatic"])
    test_blend_lc[mask_asy_test] = (LC_WC * r_cat_test[mask_asy_test]
                                     + LC_WM * r_mlp_test[mask_asy_test]
                                     + LC_WX * r_xgb_test[mask_asy_test])
    print(f"Applied local correction to {mask_asy_test.sum()} test samples.")

print("\nOOF AUC summary after Phase 1:")
for name, oof in [("R1 equal blend", oof_blend_init),
                  ("R1 opt blend",   oof_blend_opt),
                  ("R1 + local corr",oof_blend_lc)]:
    print(f"  {name:<25}: {roc_auc_score(y_comp, oof):.6f}")


## 15. Phase 2A — Soft Noise Weighting

Compute per-sample weights using `w = 1 - alpha * normalised(|blend - y|)`.  
Noisier samples are **down-weighted**, not removed.


In [ ]:
print("="*60)
print("Phase 2A: Soft Noise Weight Computation")
print("="*60)

# Use Phase-1 local-corrected blend as reference
p_ref = oof_blend_lc

sample_weights_full = make_soft_weights(
    p_ref,
    y_comp,
    alpha = CFG['noise_alpha'],
    min_w = CFG['noise_min_w'],
)

print(f"alpha = {CFG['noise_alpha']},  min_weight = {CFG['noise_min_w']}")
print(f"Weight stats:  min={sample_weights_full.min():.3f}  "
      f"mean={sample_weights_full.mean():.3f}  max={sample_weights_full.max():.3f}")
print(f"Samples w < 0.85: {(sample_weights_full < 0.85).sum()} ({(sample_weights_full < 0.85).mean()*100:.1f}%)")

# Quick sanity: weight vs residual direction
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(np.abs(p_ref - y_comp.values), sample_weights_full,
                alpha=0.3, s=8, c='steelblue')
axes[0].set_xlabel("|blend - y|"); axes[0].set_ylabel("sample weight")
axes[0].set_title("Noise score vs sample weight")

axes[1].hist(sample_weights_full, bins=40, color='coral', edgecolor='white')
axes[1].set_xlabel("sample weight"); axes[1].set_title("Sample weight distribution")
plt.tight_layout()
plt.savefig("soft_weights_analysis.png", dpi=100)
plt.show()
print("Saved: soft_weights_analysis.png")


## 16. Phase 2B — CatBoost Round 2 (Soft Noise Weights)

Re-train CatBoost applying per-sample weights from Phase 2A.


In [ ]:
print("="*60)
print("Phase 2B: CatBoost Round 2 — Soft Noise Weighting")
print("="*60)

oof_cat2  = np.zeros(len(X_comp), dtype="float32")
test_cat2 = np.zeros(len(X_test_comp), dtype="float32")

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_comp, y_comp), start=1):
    print(f"[CatBoost R2] Fold {fold}/{N_FOLDS}")
    X_tr, X_val = X_comp.iloc[tr_idx], X_comp.iloc[val_idx]
    y_tr, y_val = y_comp.iloc[tr_idx], y_comp.iloc[val_idx]
    w_tr = sample_weights_full[tr_idx]  # apply weights to train only

    train_pool = Pool(X_tr, label=y_tr, cat_features=cat_features_idx, weight=w_tr)
    valid_pool = Pool(X_val, label=y_val, cat_features=cat_features_idx)
    test_pool  = Pool(X_test_comp, cat_features=cat_features_idx)

    model = CatBoostClassifier(random_seed=RANDOM_STATE + fold + 100, **_CAT_PARAMS)
    model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

    oof_cat2[val_idx]  = model.predict_proba(valid_pool)[:, 1].astype("float32")
    test_cat2         += (model.predict_proba(test_pool)[:, 1] / N_FOLDS).astype("float32")

cat2_oof_auc = roc_auc_score(y_comp, oof_cat2)
print(f"\n[CatBoost R2] OOF AUC = {cat2_oof_auc:.6f}")
print(f"[CatBoost R1] OOF AUC = {roc_auc_score(y_comp, oof_cat):.6f}  (reference)")


## 17. Phase 2C — XGBoost Round 2 (Soft Noise Weights)

In [ ]:
print("="*60)
print("Phase 2C: XGBoost Round 2 — Soft Noise Weighting")
print("="*60)

oof_xgb2  = np.zeros(len(X_comp), dtype="float32")
test_xgb2 = np.zeros(len(X_test_num), dtype="float32")

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_comp_num, y_comp), start=1):
    print(f"[XGBoost R2] Fold {fold}/{N_FOLDS}")
    X_tr, X_val = X_comp_num.iloc[tr_idx], X_comp_num.iloc[val_idx]
    y_tr, y_val = y_comp.iloc[tr_idx], y_comp.iloc[val_idx]
    w_tr = sample_weights_full[tr_idx]

    model = xgb.XGBClassifier(**_XGB_PARAMS)
    model.fit(X_tr, y_tr, sample_weight=w_tr,
              eval_set=[(X_val, y_val)], verbose=False)

    oof_xgb2[val_idx]  = model.predict_proba(X_val)[:, 1].astype("float32")
    test_xgb2         += (model.predict_proba(X_test_num)[:, 1] / N_FOLDS).astype("float32")

xgb2_oof_auc = roc_auc_score(y_comp, oof_xgb2)
print(f"\n[XGBoost R2] OOF AUC = {xgb2_oof_auc:.6f}")
print(f"[XGBoost R1] OOF AUC = {roc_auc_score(y_comp, oof_xgb):.6f}  (reference)")


## 18. Phase 2D — Final Re-Blend (Noise-Aware Models)

Grid search again over Round-2 predictions to find the optimal final blend.


In [ ]:
print("="*60)
print("Phase 2D: Final Re-Blend (noise-aware models)")
print("="*60)

# Pool of all model OOFs for final blend candidate search
all_oofs = {
    "cat1":  oof_cat,
    "mlp":   oof_realmlp,
    "xgb1":  oof_xgb,
    "cat2":  oof_cat2,
    "xgb2":  oof_xgb2,
}
all_tests = {
    "cat1":  test_cat,
    "mlp":   test_realmlp,
    "xgb1":  test_xgb,
    "cat2":  test_cat2,
    "xgb2":  test_xgb2,
}

print("\nIndividual model OOF AUCs:")
for name, oof in all_oofs.items():
    print(f"  {name:<8}: {roc_auc_score(y_comp, oof):.6f}")

# ── 5-model grid search (coarser grid for speed) ────────────────────────────
# Strategy: exhaustive over (cat1+cat2) combined as single catboost bucket,
# then fine-tune full 5-model below.

# Step 1: Best 3-model blend using noise-aware models
r2_cat2 = rank01(oof_cat2)
r2_mlp  = rank01(oof_realmlp)
r2_xgb2 = rank01(oof_xgb2)

best2_auc = 0
best2_ws  = (1/3, 1/3, 1/3)
for wc in ws:
    for wm in ws:
        wx = round(1.0 - wc - wm, 8)
        if wx < -1e-6 or wx > 1.0 + 1e-6:
            continue
        wx = max(0.0, min(1.0, wx))
        blend = wc * r2_cat2 + wm * r2_mlp + wx * r2_xgb2
        auc = roc_auc_score(y_comp, blend)
        if auc > best2_auc:
            best2_auc = auc
            best2_ws = (wc, wm, wx)

B2_WC, B2_WM, B2_WX = best2_ws
print(f"\nBest noise-aware 3-model blend:")
print(f"  Cat2={B2_WC:.2f}  MLP={B2_WM:.2f}  XGB2={B2_WX:.2f}  →  OOF AUC={best2_auc:.6f}")

oof_blend_noise  = B2_WC * r2_cat2 + B2_WM * r2_mlp + B2_WX * r2_xgb2
test_blend_noise = (B2_WC * rank01(test_cat2)
                    + B2_WM * rank01(test_realmlp)
                    + B2_WX * rank01(test_xgb2))

# Step 2: Best 5-model blend (coarser grid)
step5 = 0.1
ws5 = np.arange(0.0, 1.01, step5)
r_all = [rank01(oof_cat), rank01(oof_realmlp), rank01(oof_xgb),
         rank01(oof_cat2), rank01(oof_xgb2)]
names5 = ["cat1", "mlp", "xgb1", "cat2", "xgb2"]

best5_auc = 0; best5_ws = None
# Sample random weights for 5-model (grid too large)
rng = np.random.default_rng(RANDOM_STATE)
W_rand = rng.dirichlet(np.ones(5), size=5000)
for w in W_rand:
    blend = sum(w[i] * r_all[i] for i in range(5))
    auc = roc_auc_score(y_comp, blend)
    if auc > best5_auc:
        best5_auc = auc
        best5_ws = w.copy()

print(f"\nBest 5-model blend (random search, n=5000):")
for n, wv in zip(names5, best5_ws):
    print(f"  {n}: {wv:.3f}")
print(f"  OOF AUC = {best5_auc:.6f}")

oof_blend_5m  = sum(best5_ws[i] * r_all[i] for i in range(5))
test_blend_5m = sum(best5_ws[i] * rank01(all_tests[names5[i]]) for i in range(5))


## 19. Phase 3 — Regime-Aware Evaluation

Compare all blend candidates across regimes.  
This informs whether a regime-specific local correction is warranted.


In [ ]:
print("="*60)
print("Phase 3: Regime-Aware Evaluation")
print("="*60)

candidates_oof = {
    "Cat1":          oof_cat,
    "RealMLP":       oof_realmlp,
    "XGB1":          oof_xgb,
    "Cat2(noisy)": oof_cat2,
    "XGB2(noisy)": oof_xgb2,
    "Ph1-EqBlend":   oof_blend_init,
    "Ph1-OptBlend":  oof_blend_opt,
    "Ph1+LocCorr":   oof_blend_lc,
    "Ph2-3model":    oof_blend_noise,
    "Ph2-5model":    oof_blend_5m,
}

# Subgroup masks (update hard-example with final blend)
residual_final = np.abs(oof_blend_5m - y_comp.values.astype(float))
mask_hard_final = residual_final >= np.percentile(residual_final, 92)
subgroup_masks_final = {**subgroup_masks, "hard_8pct(final)": mask_hard_final}

print()
print_auc_table(y_comp, candidates_oof, subgroup_masks_final)


## 20. Select Best Blend & Apply Final Local Correction

Pick the highest OOF-AUC candidate as base, then apply regime-specific local corrections.


In [ ]:
print("="*60)
print("Final blend selection & local correction")
print("="*60)

# Auto-select highest OOF-AUC candidate
best_final_name = max(candidates_oof, key=lambda n: roc_auc_score(y_comp, candidates_oof[n]))
best_final_auc  = roc_auc_score(y_comp, candidates_oof[best_final_name])
print(f"Auto-selected: {best_final_name}  (OOF AUC={best_final_auc:.6f})")

# Map selected to test blend
_test_map = {
    "Cat1":         test_cat,
    "RealMLP":      test_realmlp,
    "XGB1":         test_xgb,
    "Cat2(noisy)": test_cat2,
    "XGB2(noisy)": test_xgb2,
    "Ph1-EqBlend":  np.mean([rank01(test_cat), rank01(test_realmlp), rank01(test_xgb)], axis=0),
    "Ph1-OptBlend": test_blend_opt,
    "Ph1+LocCorr":  test_blend_lc,
    "Ph2-3model":   test_blend_noise,
    "Ph2-5model":   test_blend_5m,
}
test_final_base = _test_map[best_final_name]
oof_final_base  = candidates_oof[best_final_name].copy()

# ── Optional: 2nd-pass local correction on cp=ASY using best noise-aware models ──
oof_final  = oof_final_base.copy()
test_final = test_final_base.copy()

if _do_local:
    # Re-search over noise-aware models for ASY local correction
    r2_sets = {
        "cat1": rank01(oof_cat),  "mlp": rank01(oof_realmlp),
        "xgb1": rank01(oof_xgb),  "cat2": rank01(oof_cat2),  "xgb2": rank01(oof_xgb2),
    }
    best_lc2_auc = roc_auc_score(y_comp, oof_final)
    best_lc2_combo = None

    for n1, r1 in r2_sets.items():
        for n2, r2 in r2_sets.items():
            for alpha_lc in np.arange(0.0, 1.01, 0.1):
                oof_tmp = oof_final.copy()
                oof_tmp[mask_asy] = alpha_lc * r1[mask_asy] + (1 - alpha_lc) * r2[mask_asy]
                auc = roc_auc_score(y_comp, oof_tmp)
                if auc > best_lc2_auc:
                    best_lc2_auc = auc
                    best_lc2_combo = (n1, n2, alpha_lc)

    if best_lc2_combo:
        n1, n2, alpha_lc = best_lc2_combo
        print(f"\nFinal local correction (cp=ASY):  {alpha_lc:.1f}*{n1} + {1-alpha_lc:.1f}*{n2}")
        print(f"OOF AUC: {best_lc2_auc:.6f}  (before: {roc_auc_score(y_comp, oof_final):.6f})")

        r1_test = rank01(_test_map.get(n1.replace("cat1", "Cat1").replace("mlp", "RealMLP")
                                         .replace("xgb1", "XGB1").replace("cat2", "Cat2(noisy)")
                                         .replace("xgb2", "XGB2(noisy)"), test_final))
        r2_test = rank01(_test_map.get(n2.replace("cat1", "Cat1").replace("mlp", "RealMLP")
                                         .replace("xgb1", "XGB1").replace("cat2", "Cat2(noisy)")
                                         .replace("xgb2", "XGB2(noisy)"), test_final))

        oof_final[mask_asy] = (alpha_lc * r2_sets[n1][mask_asy]
                               + (1 - alpha_lc) * r2_sets[n2][mask_asy])
        if mask_asy_test.any():
            test_final[mask_asy_test] = (alpha_lc * r1_test[mask_asy_test]
                                         + (1 - alpha_lc) * r2_test[mask_asy_test])
    else:
        print("Local correction did not improve OOF AUC — using base blend.")

print(f"\n★ FINAL OOF AUC = {roc_auc_score(y_comp, oof_final):.6f}")


## 21. Submission Candidates

In [ ]:
submission_candidates = {
    "cat1_only":       test_cat,
    "xgb1_only":       test_xgb,
    "ph1_opt_blend":   test_blend_opt,
    "ph1_local_corr":  test_blend_lc,
    "ph2_3model":      test_blend_noise,
    "ph2_5model":      test_blend_5m,
    "final":           test_final,
}

oof_candidates = {
    "cat1_only":       oof_cat,
    "xgb1_only":       oof_xgb,
    "ph1_opt_blend":   oof_blend_opt,
    "ph1_local_corr":  oof_blend_lc,
    "ph2_3model":      oof_blend_noise,
    "ph2_5model":      oof_blend_5m,
    "final":           oof_final,
}

print("OOF AUC for all candidates:")
for name in submission_candidates:
    auc = roc_auc_score(y_comp, oof_candidates[name])
    print(f"  {name:<25}: {auc:.6f}")

selected_submission_name = "final"
print(f"\nSelected: {selected_submission_name}")


## 22. Save CSVs

In [ ]:
for name, pred in submission_candidates.items():
    sub_df = pd.DataFrame({
        "id": test_ids.values if hasattr(test_ids, "values") else test_ids,
        "Heart Disease": np.clip(pred.astype(np.float64), 0.0, 1.0),
    })
    fname = f"submission_{name}.csv"
    sub_df.to_csv(fname, index=False)
    print(f"Saved: {fname}  shape={sub_df.shape}")

# Final submission
final_pred = np.clip(
    submission_candidates[selected_submission_name].astype(np.float64),
    0.0, 1.0,
)
submission = pd.DataFrame({
    "id": test_ids.values if hasattr(test_ids, "values") else test_ids,
    "Heart Disease": final_pred,
})
submission.to_csv("submission.csv", index=False)
print(f"\nSaved final: submission.csv  ({selected_submission_name})")


## 23. Final Summary

In [ ]:
print("="*60)
print("FINAL SUMMARY")
print("="*60)

print("\nIndividual models:")
for name, oof in [("CatBoost R1", oof_cat), ("RealMLP", oof_realmlp),
                   ("XGBoost R1", oof_xgb), ("CatBoost R2", oof_cat2),
                   ("XGBoost R2", oof_xgb2)]:
    print(f"  {name:<15}: {roc_auc_score(y_comp, oof):.6f}")

print("\nBlend progression:")
for name, oof in [("Ph1 equal", oof_blend_init), ("Ph1 opt", oof_blend_opt),
                   ("Ph1+LocCorr", oof_blend_lc), ("Ph2 3-model", oof_blend_noise),
                   ("Ph2 5-model", oof_blend_5m), ("FINAL", oof_final)]:
    print(f"  {name:<15}: {roc_auc_score(y_comp, oof):.6f}")

print("\nSaved files:")
for name in submission_candidates:
    print(f"  submission_{name}.csv")
print("  submission.csv  (→ final)")
